<a href="https://colab.research.google.com/github/Ireneunav/TFM_MCD25/blob/main/Funcion_gradillas_estaciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extracción de datos de EE

Earth Engine (EE) es un archivo de datos de Google donde los usuarios pueden visualizar, editar y extraer datos de diferentes satélites de la NASA. Para acceder remotamente a este archivo se debe tener acceso a un proyecto de Google Cloud. Una vez que se tiene acceso se deben seguir unos pasos para acceder a dicho proyecto desde Colab o desde VS Code.

## Acceso a Earth Engine

In [ ]:
import ee
import geemap
from google.colab import drive
drive.mount('/content/drive')

ee.Authenticate()
ee.Initialize(project = 'datakorea')

Mounted at /content/drive


Después de acceder al proyecto se pueden buscar los datos que se necesitan y analizarlos, aunque en este caso vamos a preparar una función que permita automatizar todo el proceso de extracción de datos y unificarlos en una tabla. Para ello desarrollamos esta tabla `cheat`, que facilita la apertura de los datos reuniendo todo lo necesario en ella.

## Preparación de la tabla de automatización

In [ ]:
import pandas as pd
import numpy as np

variables_ee = pd.read_csv('drive/MyDrive/Máster/TFM/columnas_EE.csv', header=None) # las variables se encuentran en una tabla diferente y serán unificadas con la tabla cheat "manualmente"
cheat = pd.read_csv('drive/MyDrive/Máster/TFM/columnas_EE_cheat_automatizacion.csv') # la tabla `cheat` también fue construida a mano y se importa sin bandas y con todas las variables de
  # Google Earth Engine que indica el artículo de Kwon D et al. (2022)
cheat['Columnas'] = None

# Duplicaremos las filas de cheat que contengan variables "totales" para no agruparlas por medias sino por sumas
totales = []
for i, banda in enumerate(variables_ee.iloc[:][0]):
  if 'total' in str(banda):
    totales.append(i)
bandas_totales = variables_ee.iloc[totales]
bandas_sin_totales = variables_ee.drop(totales)

# Fila por fila recorreremos la tabla y agregaremos las variables correspondientes de cada conjunto de datos en forma de lista a la columna `Columnas`
for n in range(0,cheat.shape[0]):
    if n == 0:
        idx = np.where(bandas_sin_totales.index == cheat['Columnas_EE_fin'].iloc[n])[0][0]
        cheat.at[n, 'Columnas'] = bandas_sin_totales.iloc[0:idx, 0].tolist()
        idx_last = idx
    elif n == cheat.shape[0]-1:
        idx = bandas_sin_totales.shape[0]
        cheat.at[n, 'Columnas'] = bandas_sin_totales.iloc[idx_last:idx, 0].tolist()
    else:
        idx = np.where(bandas_sin_totales.index == cheat['Columnas_EE_fin'].iloc[n])[0][0]
        cheat.at[n, 'Columnas'] = bandas_sin_totales.iloc[idx_last:idx, 0].tolist() #inicio es la última (incluida) y el final es el actual (no incluido)
        idx_last = idx

# Voy a copiar las filas de cheat en las que vayan las bandas de totales, y para automatizarlo voy a buscar por la columna Columnas_EE
for m in range(0, bandas_totales.shape[0]):
  fila = np.where(cheat['Columnas_EE_fin'] > bandas_totales.index[m])[0].min()
  fila_df = cheat.iloc[[fila]]
  fila_df.at[fila, 'Columnas'] = bandas_totales.iloc[m][0]
  cheat = pd.concat([cheat[:fila+1], fila_df, cheat.iloc[fila+1:]], ignore_index=True)

# Algunos de los conjuntos no los utilizaremos y los eliminamos
mcd43 = np.where(cheat['Modelo'] == 'MCD43A3')[0]
glob_cov = np.where(cheat['Fuente'] == 'Globcover')[0]
cheat = cheat.drop(mcd43)
cheat = cheat.drop(glob_cov)
cheat = cheat.drop(columns='Columnas_EE_fin')
cheat = cheat.reset_index(drop=True) # Reseteamos los índices para que sean enteros seguidos
print(cheat.head())

  Fuente     Modelo                        Nombre Fecha_inicio   Fecha_fin  \
0  ECMWF       ERA5            ECMWF/ERA5/MONTHLY   1979-01-01  2020-06-01   
1  ECMWF       ERA5            ECMWF/ERA5/MONTHLY   1979-01-01  2020-06-01   
2  ECMWF  ERA5-Land  ECMWF/ERA5_LAND/MONTHLY_AGGR   1950-02-01  2026-05-01   
3  ECMWF  ERA5-Land  ECMWF/ERA5_LAND/MONTHLY_AGGR   1950-02-01  2026-05-01   
4  MODIS    MOD09A1             MODIS/061/MOD09A1   2000-02-18  2026-05-25   

   Columnas_EE_fin                                           Columnas  
0                8  [mean_2m_air_temperature, minimum_2m_air_tempe...  
1                8                                total_precipitation  
2               19  [dewpoint_temperature_2m, leaf_area_index_high...  
3               19                            total_precipitation_sum  
4               29  [sur_refl_b01, sur_refl_b02, sur_refl_b03, sur...  


Esta tabla contiene la fuente de los datos, el modelo del satélite que recogió los datos, el enlace que hará falta para abrirlos desde EE, la fecha de inicio y de fin de los datos y las columnas o bandas que necesitaremos en el proyecto.

In [ ]:
cheat

,Fuente,Modelo,Nombre,Fecha_inicio,Fecha_fin,Columnas_EE_fin,Columnas
0,ECMWF,ERA5,ECMWF/ERA5/MONTHLY,1979-01-01,2020-06-01,8,"[mean_2m_air_temperature, minimum_2m_air_tempe..."
1,ECMWF,ERA5,ECMWF/ERA5/MONTHLY,1979-01-01,2020-06-01,8,total_precipitation
2,ECMWF,ERA5-Land,ECMWF/ERA5_LAND/MONTHLY_AGGR,1950-02-01,2026-05-01,19,"[dewpoint_temperature_2m, leaf_area_index_high..."
3,ECMWF,ERA5-Land,ECMWF/ERA5_LAND/MONTHLY_AGGR,1950-02-01,2026-05-01,19,total_precipitation_sum
4,MODIS,MOD09A1,MODIS/061/MOD09A1,2000-02-18,2026-05-25,29,"[sur_refl_b01, sur_refl_b02, sur_refl_b03, sur..."
5,MODIS,MOD11A1,MODIS/061/MOD11A1,2000-02-24,2026-06-05,33,"[Emis_31, Emis_32, Clear_day_cov, Clear_night_..."
6,MODIS,MCD15A3H,MODIS/061/MCD15A3H,2002-07-04,2026-05-29,35,"[Fpar, Lai]"
7,MODIS,MOD13A2,MODIS/061/MOD13A2,2000-02-18,2026-05-09,37,"[NDVI, EVI]"
8,MODIS,MOD08_M3,MODIS/061/MOD08_M3,2000-02-01,2026-05-01,42,"[Aerosol_Optical_Depth_Land_Ocean_Mean_Mean, A..."
9,MODIS,MCD19A2,MODIS/061/MCD19A2_GRANULES,2000-02-24,2026-06-07,44,"[Optical_Depth_047, Optical_Depth_055]"


A continuación creamos un diccionario que contenga y permita acceder a las escalas correspondientes de cada variable de datos, de manera que según sea necesario se escalaran los registros.

In [ ]:
dicc_escalas = {
    "MOD09A1":[
        {"band_pattern": "sur_refl_.*",
         "multiply": 0.0001,
         "add": 0}
    ],
    "MOD11A1": [
        {"band_pattern": "Emis_3.",
         "multiply": 0.002,
         "add": 0.49},
         {"band_pattern": "Clear_.*_cov",
          "multiply": 0.0005,
          "add": 0}
    ],
    "MCD15A3H":[
        {"band_pattern": "Fpar",
         "multiply": 0.01,
         "add": 0},
         {"band_pattern": "Lai",
          "multiply": 0.1,
          "add": 0}
    ],
    "MOD13A2":[
        {"band_pattern": ".*VI",
         "multiply": 0.0001,
         "add": 0}
    ],
    "MOD08_M3":[
        {"band_pattern": "Aerosol_Optical_Depth_Land_.*_Mean_.*",
         "multiply": 0.001,
         "add": 0},
         {"band_pattern": "Cirrus_.*_FMean",
          "multiply": 0.0001,
          "add": 0},
          {"band_pattern": "Cloud_Optical_.*_Mean_Mean",
           "multiply": 0.001,
           "add": 0},
           {"band_pattern": "Cloud_Optical_.*_Mean_Uncertainty",
            "multiply": 0.01,
            "add": 0}
    ],
    "MCD19A2": [
        {"band_pattern": "Optical_Depth_.*",
         "multiply": 0.001,
         "add": 0},
    ],
    "Landsat7":[
        {"band_pattern": "SR_B.",
         "multiply": 2.75e-5,
         "add": -0.2},
         {"band_pattern": "ST_B6",
          "multiply": 0.00341802,
          "add": 149}]
}

Las estaciones de monitoreo y sus coordenadas están guardadas en archivos maestros, por lo que se deberán abrir los archivos maestros de los años de interés y unificarlos para tener información de todas las estaciones que se pueden haber añadido con el paso de los años.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from typing import Literal

def abrir_unir_maestros(anio_inicio: str, anio_final_inc: str, ruta_general: str, preguntar_satisfaccion: Literal['si', 'no']='no'):

  anios = np.arange(anio_inicio, (pd.Timestamp(anio_final_inc) + pd.DateOffset(years=1)),
                    dtype='datetime64[Y]')
  resultados = {}
  for anio in anios:
    base = Path(f'{ruta_general}/{anio}')
    if not base.exists():
      raise ValueError(f'El directorio {base} no existe')
      return []

    claves = ['maestro', 'metainformacion', 'magnitudes', 'estaciones']
    coincidencias_maestro = [
        p for p in base.glob("*")
        if p.is_file() and any(clave in p.name.lower() for clave in claves)
    ]
    if not coincidencias_maestro:
      print(f'No se ha encontrado archivo maestro en {base}')
      eleccion_maestro = []
    elif len(coincidencias_maestro) == 1:
      print(f'Se encontró {coincidencias_maestro}')
      if preguntar_satisfaccion == 'no':
        satis = 'si'
      else:
        satis = input('Está satisfecha con el resultado? (si/no)').lower()

      if satis in ['sí', 'si', 'yes', 'y', 'bai']:
        eleccion_maestro = coincidencias_maestro[0]
    else:
      print(f'Se encontraron varias coincidencias')
      for i, arch in enumerate(coincidencias_maestro):
        print(f'{i}. {arch}')
      if preguntar_satisfaccion == 'no':
        satis = 'si'
      else:
        satis = input('Está satisfecha con el resultado? (si/no)').lower()
      if satis in ['sí', 'si', 'yes', 'y', 'bai']:
        eleccion = input('Introduzca el índice del archivo tal cual como sale en la pantalla')
        eleccion_maestro = coincidencias_maestro[eleccion]
    resultados[str(anio)] = eleccion_maestro
  print(resultados)

  estaciones = pd.DataFrame()
  for anio in anios:
    if resultados[str(anio)].suffix == '.csv':
        est_anio = pd.read_csv(resultados[str(anio)], sep=';', encoding='iso-8859-1')
    elif resultados[str(anio)].suffix in ['.xlsx', '.xls']:
        est_anio = pd.read_excel(resultados[str(anio)], sheet_name=f'Estaciones evaluación {anio}')
    else:
      raise ValueError("Extensión o archivo maestro no compatibles con apertura automática")

    estaciones = pd.concat([estaciones, est_anio], ignore_index = True).drop_duplicates(ignore_index=True)
  return estaciones



In [ ]:
maestro = abrir_unir_maestros('2018', '2019', 'drive/MyDrive/Máster/TFM/Irene Iribarren TFM (shared)/02_data/ICA_abiertos/')
provincias_quitar = [ 'BALEARS (ILLES)', 'CEUTA', 'PALMAS (LAS)', 'MELILLA', 'SANTA CRUZ DE TENERIFE']
comunidad_quitar = ['BALEARES (ISLAS)', 'CANARIAS']
maestro = maestro[~maestro["N_CCAA"].isin(comunidad_quitar)].copy()
maestro = maestro.reset_index(drop=True)
maestro

Se encontró [PosixPath('drive/MyDrive/Máster/TFM/Irene Iribarren TFM (shared)/02_data/ICA_abiertos/2018/metainformacion_2018_tcm30-501408.xlsx')]
Se encontró [PosixPath('drive/MyDrive/Máster/TFM/Irene Iribarren TFM (shared)/02_data/ICA_abiertos/2019/metainformacion_2019_tcm30-513561.xlsx')]
{'2018': PosixPath('drive/MyDrive/Máster/TFM/Irene Iribarren TFM (shared)/02_data/ICA_abiertos/2018/metainformacion_2018_tcm30-501408.xlsx'), '2019': PosixPath('drive/MyDrive/Máster/TFM/Irene Iribarren TFM (shared)/02_data/ICA_abiertos/2019/metainformacion_2019_tcm30-513561.xlsx')}


,COD_LOCAL,PROVINCIA,MUNICIPIO,ESTACION,COD_ESTACION_DEM,N_RED,NOMBRE,FECHA_INI,FECHA_FIN,LATITUD_G,...,ALTITUD,N_CCAA,N_PROVINCIA,N_MUNICIPIO,TIPO_ESTACION,TIPO_AREA,TIPO_SUBAREA_RURAL,EST,ZONA,DIRECCION
0,1022001,1,22,1,ES1672A,CCAA País Vasco,EL CIEGO,2004-05-10,NaT,42.51833,...,480,PAÍS VASCO,ÁLAVA,ELCIEGO,TRAFICO,SUBURBANA,NaN,ST,RESIDENCIAL,"C/GABRIEL CELAYA 8, ELCIEGO"
1,1036004,1,36,4,ES1349A,CCAA País Vasco,LLODIO,1994-01-01,NaT,43.14407,...,122,PAÍS VASCO,ÁLAVA,LLODIO,TRAFICO,SUBURBANA,NaN,ST,RESIDENCIAL/COMERCIAL,"C/ LAMUZA, S/N"
2,1051001,1,51,1,ES1544A,CCAA País Vasco,AGURAIN,1998-01-01,NaT,42.84900,...,594,PAÍS VASCO,ÁLAVA,SALVATIERRA O AGURAIN,FONDO,SUBURBANA,NaN,SF,RESIDENCIAL/INDUSTRIAL,CUARTEL DE LA ERTZANTZA
3,1055001,1,55,1,ES1489A,CCAA País Vasco,VALDEREJO,1998-01-01,NaT,42.87520,...,911,PAÍS VASCO,ÁLAVA,VALDEGOVÍA,FONDO,RURAL,REMOTA,RFREM,NATURALEZA,PARQUE NATURAL DE VALDEREJO
4,1059008,1,59,8,ES1502A,CCAA País Vasco,AVENIDA GASTEIZ,1998-01-01,NaT,42.85480,...,517,PAÍS VASCO,ÁLAVA,VITORIA-GASTEIZ,TRAFICO,URBANA,NaN,UT,RESIDENCIAL/COMERCIAL,"AVDA. GASTEIZ, 93"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
537,28120001,28,120,1,ES2093A,CCAA Madrid,PUERTO DE COTOS,2019-01-01,NaT,40.82510,...,1200,MADRID,MADRID,RASCAFRÍA,FONDO,RURAL,REMOTA,RFREM,NATURALEZA,Centro de Visitantes de ?Pe?alara?. Ctra. M-60...
538,31232003,31,232,3,ES2094A,CCAA Navarra,TUDELA II,2019-01-01,NaT,42.06150,...,262,NAVARRA (COMUNIDAD FORAL),NAVARRA,TUDELA,FONDO,URBANA,NaN,UF,RESIDENCIAL/COMERCIAL/INDUSTRIAL,C/ Fernando Remacha
539,40040001,40,40,1,ES2098A,CCAA Castilla y León,CANTALEJO,2018-01-01,NaT,41.25590,...,964,CASTILLA Y LEÓN,SEGOVIA,CANTALEJO,TRAFICO,URBANA,NaN,UT,DESCONOCIDO,NaN
540,46250054,46,250,54,ES2095A,CCAA Com. Valenciana,VALÈNCIA-CENTRE,2018-10-15,NaT,39.44833,...,2,COMUNIDAD VALENCIANA,VALENCIA,VALENCIA,TRAFICO,URBANA,NaN,UT,DESCONOCIDO,NaN


Los datos obtenidos de AEMET para la climatología tienen un formato especial para las coordenadas de las estaciones: la latitud se expresa como una secuencia de seis números (ggmmss), que corresponden a los grados, minutos y segundo; sin embargo, la longitud es una secuencia de siete números (ggmmssA), donde el último número es 1 si la coordenada corresponde al este y 2 si corresponde al oeste. Para situar correctamente estas estaciones en un mapa, debemos ponerlas en un formato compatible con nuestro programa, que será formato decimal. Estas funciones convierten estas coordenadas a ese formato decimal con el signo correspondiente en el caso de la longitud.

In [ ]:
# maestro = pd.read_csv('drive/MyDrive/Máster/TFM/Irene Iribarren TFM (shared)/02_data/Climatology/2024/Maestro_Climatologico_2024.csv', sep=';', encoding='iso-8859-1')
# print(maestro.head())
"""
Para las estaciones de ICA no hace falta convertir las coordenadas porque ya están en formato decimal,
pero para las estaciones de climatología de AEMET sí que es necesario cambiar el formato de las coordenadas.
"""
"""
def convertir_longitud(columna_longitud):
    columna_longitud = pd.DataFrame(columna_longitud)
    nombre = columna_longitud.columns.values[0]
    for n in range(len(columna_longitud)):
        num_str = str(int(columna_longitud.iloc[n].values[0])).zfill(7)
        signo = num_str[-1]
        grados = float(num_str[-7:-5])
        minutos = float(num_str[-5:-3])
        segundos = float(num_str[-3:-1])
        # print(num_str, '-->', grados, minutos, segundos, signo)

        if signo == '2':
            columna_longitud.loc[n, nombre] = (-1) * (grados + minutos/60 + segundos/3600)
        elif signo == '1':
            columna_longitud.loc[n, nombre] = grados + minutos/60 + segundos/3600
        else:
            raise ValueError(f"El número en la fila {n} no se corresponde con el formato de longitud (terminar en 1 o 2)")
    return columna_longitud

def convertir_latitud(columna_latitud):
    columna_latitud = pd.DataFrame(columna_latitud)
    nombre = columna_latitud.columns.values[0]
    for i in range(len(columna_latitud)):
        num_str = str(int(columna_latitud.iloc[i].values[0])).zfill(7)
        grados = float(num_str[-6:-4])
        minutos = float(num_str[-4:-2])
        segundos = float(num_str[-2:])
        # print(num_str, '-->', grados, minutos, segundos)
        columna_latitud.loc[i, nombre] = grados + minutos/60 + segundos/3600
    return columna_latitud

maestro['LATITUD'] = convertir_latitud(maestro['LATITUD'])
maestro['LONGITUD'] = convertir_longitud(maestro['LONGITUD'])
print(maestro.head())"""

'\nPara las estaciones de ICA no hace falta convertir las coordenadas porque ya están en formato decimal.\n\ndef convertir_longitud(columna_longitud):\n    columna_longitud = pd.DataFrame(columna_longitud)\n    nombre = columna_longitud.columns.values[0]\n    for n in range(len(columna_longitud)):\n        num_str = str(int(columna_longitud.iloc[n].values[0])).zfill(7)\n        signo = num_str[-1]\n        grados = float(num_str[-7:-5])\n        minutos = float(num_str[-5:-3])\n        segundos = float(num_str[-3:-1])\n        # print(num_str, \'-->\', grados, minutos, segundos, signo)\n\n        if signo == \'2\':\n            columna_longitud.loc[n, nombre] = (-1) * (grados + minutos/60 + segundos/3600)\n        elif signo == \'1\':\n            columna_longitud.loc[n, nombre] = grados + minutos/60 + segundos/3600\n        else:\n            raise ValueError(f"El número en la fila {n} no se corresponde con el formato de longitud (terminar en 1 o 2)")\n    return columna_longitud\n\nd

En ciertas ocasiones puede ser que necesitemos una tabla en formato `GeoDataFrame` en lugar de `DataFrame`.

In [ ]:
import geopandas as gpd
maestro_geo = gpd.GeoDataFrame(maestro)
maestro_geo.columns = maestro_geo.columns.str.lower()
maestro_geo = maestro_geo[['cod_local', 'provincia', 'municipio', 'estacion']] #, 'latitud_g', 'longitud_g'
maestro_geo['codigo_ine'] = (maestro_geo['provincia'].astype(str).str.zfill(2) +
                             maestro_geo['municipio'].astype(str).str.zfill(3))
for n in maestro_geo.columns:
  if maestro_geo[n].dtype == 'O':
    maestro_geo[n] = maestro_geo[n].astype(np.int64)
# maestro_geo['cod_local'] = maestro_geo['cod_local'].astype(object)
# maestro_geo['codigo_ine'] = maestro_geo['codigo_ine'].astype(object)
columnas = maestro_geo.columns
maestro_geo = maestro_geo.rename(
    columns={
        "provincia": "cod_provincia",
        "municipio": "cmun"
    }
)

print(maestro_geo.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 542 entries, 0 to 541
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   cod_local      542 non-null    int64
 1   cod_provincia  542 non-null    int64
 2   cmun           542 non-null    int64
 3   estacion       542 non-null    int64
 4   codigo_ine     542 non-null    int64
dtypes: int64(5)
memory usage: 21.3 KB
None


In [ ]:
from typing import Literal
import geopandas as gpd
import geemap
import ee
import numpy as np
import pandas as pd

def extraccion_datos_estaciones(directorio_mapa, dejar_quitar: Literal['dejar', 'quitar', None], comunidades: str, cheat: pd.DataFrame,
                   fecha_inicio, fecha_final, frecuencia: Literal['anual', 'mensual'], sin_fecha: Literal['ignorar_fecha', 'parar'],
                   escalas, estaciones: pd.DataFrame):
  #### Abrir el mapa de comunidades
  import geopandas as gpd

  gdf = gpd.read_file(directorio_mapa)
  if dejar_quitar == "quitar":
    gdf_rec = gdf[~gdf["CCAA"].isin(comunidades)].copy() #eliminar las provincias dadas
  elif dejar_quitar == "dejar":
    gdf_rec = gdf[gdf["CCAA"].isin(comunidades)].copy() #dejar las provincias dadas
  elif dejar_quitar == None:
    gdf_rec = gdf #no hacer nada y dejar todo el mapa dado
  gdf_rec["geometry"] = gdf_rec["geometry"].buffer(0)
  peninsula_geom = gdf_rec.geometry.union_all()

  #### Recortar los datos a España
  peninsula_gdf = gpd.GeoDataFrame(
    {'name': ['España peninsular']},
    geometry=[peninsula_geom],
    crs=gdf_rec.crs)

  #### Seleccionar las cuadrículas que tienen una estación de monitoreo
  peninsula_fc = geemap.geopandas_to_ee(peninsula_gdf)
  peninsula = peninsula_fc.geometry()
  maestro['LATITUD_G'] = pd.to_numeric(maestro['LATITUD_G'], errors='coerce')
  maestro['LONGITUD_G'] = pd.to_numeric(maestro['LONGITUD_G'], errors='coerce')
  maestro['COD_LOCAL'] = (maestro['COD_LOCAL'].astype(str))

  features = [ee.Feature(ee.Geometry.Point(maestro['LONGITUD_G'][n], maestro['LATITUD_G'][n]),
                          {"id": maestro['COD_LOCAL'][n], "lat": maestro['LATITUD_G'][n], "lon": maestro['LONGITUD_G'][n]}) for n in range(0, maestro.shape[0])]

  estaciones_fc = ee.FeatureCollection(features)
  estaciones_fc = estaciones_fc.filterBounds(peninsula)

  proj = ee.Projection("EPSG:3035")
  grid = peninsula.coveringGrid(proj, 1000)

  grid_estaciones = grid.filterBounds(estaciones_fc)
  estaciones_fc = estaciones_fc.map(lambda f: f.set("station_id", f.get('COD_LOCAL')))

  # Hay muchos datos que tienen que ser escalados. Para eso usamos una funcion y un diccionario que hemos hecho antes.
  def apply_scale_factors(image, rules):
    for rule in rules:
      scaled = (
          image.select(rule["band_pattern"]).multiply(rule.get("multiply", 1)).add(rule.get("add",0))
      )
      image = image.addBands(scaled, None, True)
    return image

  # Hay que "crear" el vector que recorreremos con las fechas
  if frecuencia == 'anual':
    final = pd.Timestamp(fecha_final) + pd.DateOffset(years=1)
    anios = np.arange(fecha_inicio, final, dtype='datetime64[Y]')
    pasos_og = [f'{x}-01-01' for x in anios]
  else:
    final = pd.Timestamp(fecha_final) + pd.DateOffset(months=1)
    meses = np.arange(fecha_inicio, final, dtype='datetime64[M]')
    pasos_og = [f'{x}-01' for x in meses]

  for actual in range(0, cheat.shape[0]):
    pasos = pasos_og.copy()
    n = 0
    m = -1
    fecha_i = pasos[n]
    fecha_f = pasos[m]

    if sin_fecha == 'ignorar_fecha':
      while fecha_i < cheat['Fecha_inicio'][actual]:
        n += 1
        fecha_i = pasos[n]
      while fecha_f > cheat['Fecha_fin'][actual]:
        m -= 1
        fecha_f = pasos[m]
    elif sin_fecha == 'parar':
      raise ValueError(f'La fecha {fecha_inicio} no existe en los datos {cheat["Fuente"][actual]}. Por eso, el bucle se ha parado.')

    # Recortamos los pasos
    if m == -1:
      pasos = pasos[n:]
    else:
      pasos = pasos[n:m+1]

    # Ahora ya empezamos a recorrer por datos y fechas
    if len(pasos) == 1:
      steps = pasos
    else:
      steps = pasos[:-1]
    for i, pasito in enumerate(steps):

      if cheat['Fuente'][actual].strip().capitalize() == 'Copernicus': #Copernicus
        # print('Has entrado en Copernicus')
        anio = np.datetime64(pasito, '[Y]')
        img = ee.Image(f"{cheat['Nombre'][actual]}/{anio}").select(cheat['Columnas'][actual]).clip(estaciones_fc)
      elif cheat['Fuente'][actual] == 'USGS': # USGS
        # print('Has entrado en USGS')
        dataset = ee.ImageCollection(cheat['Nombre'][actual]).filterDate(pasito, pasos[i+1])
        modelo = cheat['Modelo'][actual]
        if modelo in escalas:
          reglas = escalas[modelo]
          dataset = dataset.map(lambda img: apply_scale_factors(img, reglas))
        img = dataset.mean()
        img = img.clip(estaciones_fc)
        img = img.select(cheat['Columnas'][actual])
      elif cheat['Modelo'][actual].strip().upper() == 'MCD12Q1':
        # print('Has entrado en MCD12Q1')
        anio = np.datetime64(pasito, '[Y]')
        dataset = ee.ImageCollection(cheat['Nombre'][actual]).filterDate(f'{anio}-01-01', f'{anio}-02-01')
        img = dataset.first().clip(estaciones_fc)
        img = img.select(cheat['Columnas'][actual])
      else:
        dataset = ee.ImageCollection(cheat['Nombre'][actual]).filterDate(pasito, pasos[i+1])
        modelo = cheat['Modelo'][actual]
        if modelo in escalas:
          reglas = escalas[modelo]
          dataset = dataset.map(lambda img: apply_scale_factors(img, reglas))
        img = dataset.mean().clip(estaciones_fc)
        img = img.select(cheat['Columnas'][actual])
      if cheat['Modelo'][actual] == 'MCD12Q1':
        grid_values = img.reduceRegions(
          collection = estaciones_fc,
          reducer = ee.Reducer.mode(),
          scale = 1000
        )
      elif 'total' in cheat['Columnas'][actual]:
        grid_values = img.reduceRegions(
          collection = estaciones_fc,
          reducer = ee.Reducer.sum(),
          scale = 1000
        )
      else:
        grid_values = img.reduceRegions(
          collection=estaciones_fc,
          reducer = ee.Reducer.mean(),
          scale = 1000)

      def add_date(f):
        return f.set({
          "fecha": pasito,
          "station_id": f.get("station_id")
        })

      grid_values = grid_values.map(add_date)
      df_temp = geemap.ee_to_df(grid_values)
      if i == 0:
        fecha_actual = df_temp
      else:
        fecha_actual = pd.concat([fecha_actual, df_temp], axis=0)

    if actual == 0:
      tabla_fin = fecha_actual
    else:
      tabla_fin = pd.merge(tabla_fin, fecha_actual, on=["id", 'fecha', 'lat', 'lon'], how='outer')
    # print(tabla_fin.columns)
    print(f'Datos {cheat["Modelo"][actual]} completados')
    tabla_fin = tabla_fin.drop_duplicates()
  return tabla_fin


In [ ]:
peninsular = ['Ceuta', 'Melilla', 'Illes Balears', 'Canarias']
tabla = extraccion_datos_estaciones('drive/MyDrive/Máster/TFM/provincias_spain.geojson', 'dejar', ['Comunidad de Madrid'],
                       cheat, '2016-01-01', '2021-01-01', 'mensual', 'ignorar_fecha', escalas=dicc_escalas, estaciones = maestro)
print(tabla.head())

Datos ERA5 completados
Datos ERA5 completados
Datos ERA5-Land completados
Datos ERA5-Land completados
Datos MOD09A1 completados
Datos MOD11A1 completados
Datos MCD15A3H completados
Datos MOD13A2 completados
Datos MOD08_M3 completados


Datos MCD19A2 completados
Datos water vapor completados
Datos cloud cover completados
Datos ozone completados
Datos precipitation completados
Datos Landsat7 completados
Datos Landcover completados
Datos MCD12Q1 completados
   dewpoint_2m_temperature       fecha        id       lat      lon  \
0               278.095673  2016-01-01  28005002  40.47933 -3.37795   
1               275.495636  2016-02-01  28005002  40.47933 -3.37795   
2               274.359802  2016-03-01  28005002  40.47933 -3.37795   
3               278.193665  2016-04-01  28005002  40.47933 -3.37795   
4               279.853851  2016-05-01  28005002  40.47933 -3.37795   

   maximum_2m_air_temperature  mean_2m_air_temperature  \
0                  290.816589               280.818268   
1                  290.284027               280.662384   
2                  294.663086               281.666748   
3                  295.048767               284.666290   
4                  300.933716               288.636627   

 

In [ ]:
#### Cuando hay columnas que tienen el mismo nombre y se convierten en sum_x o mean_x, y en este caso se quiere que tengan el nombre original aunque sea con _x detrás.
# sum_x --> total_precipitation ERA5
# sum_y --> total_precipitation ERA5-Land

columnas = list(tabla.columns)
sum1 = np.where(np.array(columnas) == 'sum_x')[0]
sum2 = np.where(np.array(columnas) == 'sum_y')[0]
mean1 = np.where(np.array(columnas) == 'mean_x')[0]
mean2 = np.where(np.array(columnas) == 'mean_y')[0]
mean3 = np.where(np.array(columnas) == 'mean')[0]
columnas[sum1[0]] = 'total_precipitation_x'
columnas[sum2[0]] = 'total_precipitation_y'
columnas[mean1[0]] = 'pr_wtr'
columnas[mean2[0]] = 'tcdc'
columnas[mean3[0]] = 'ozone'
tabla.columns = columnas

#ordenar las columnas como se quiera
columnas_prioritarias = ["fecha", "id", "lat", "lon"]

columnas_inicio = [
    columna
    for columna in columnas_prioritarias
    if columna in columnas
]

columnas_restantes = [
    columna
    for columna in columnas
    if columna not in columnas_inicio
]

tabla = tabla[columnas_inicio + columnas_restantes]



In [ ]:
tabla.columns

Index(['fecha', 'id', 'lat', 'lon', 'dewpoint_2m_temperature',
       'maximum_2m_air_temperature', 'mean_2m_air_temperature',
       'mean_sea_level_pressure', 'minimum_2m_air_temperature',
       'surface_pressure', 'u_component_of_wind_10m_x',
       'total_precipitation_x', 'dewpoint_temperature_2m',
       'leaf_area_index_high_vegetation', 'leaf_area_index_low_vegetation',
       'soil_temperature_level_1', 'soil_temperature_level_2',
       'soil_temperature_level_3', 'soil_temperature_level_4',
       'temperature_2m', 'u_component_of_wind_10m_y',
       'v_component_of_wind_10m', 'total_precipitation_y', 'sur_refl_b01',
       'sur_refl_b02', 'sur_refl_b03', 'sur_refl_b04', 'sur_refl_b05',
       'sur_refl_b06', 'sur_refl_b07', 'Clear_day_cov', 'Clear_night_cov',
       'Emis_31', 'Emis_32', 'Fpar', 'Lai', 'EVI', 'NDVI',
       'Aerosol_Optical_Depth_Land_Ocean_Mean_Mean',
       'Aerosol_Optical_Depth_Land_QA_Mean_Mean_470',
       'Cirrus_Fraction_SWIR_FMean',
       'Cloud_

Una vez extraídos los datos, unificados y los nombres de las columnas cambiados se puede guardar la tabla como csv para usar como se quiera.

In [ ]:
nombre = 'tabla_madrid_mensual_16-20_estICA.csv'
ruta = '/content/drive/MyDrive/Máster/TFM/Resultados/'
tabla.to_csv(f"{ruta}{nombre}", index=False, decimal = ',')
# tabla

Ahora que ya se tienen las estaciones meteorológicas y sus valores de las variables de Earth Engine extraídas, se seleccionan los datos socioeconómicos, pero únicamente los valores de los municipios o provincias que estén presentes en la tabla.

## Datos socioeconómicos

In [ ]:
# import numpy as np
# import pandas as pd
# from google.colab import drive
# drive.mount('/content/drive')

diccionario = pd.read_excel('/content/drive/MyDrive/Máster/TFM/diccionario26.xlsx')
diccionario = diccionario.map(
    lambda x: x.upper() if isinstance(x, str) else x
)
#cambiar nombres a minusculas y a nombres que coincidan con los de los datos que se abriran después
nombres = diccionario.columns.str.lower()
nombres = ['cod_ccaa' if nombre == 'codauto' else nombre for nombre in nombres]
nombres = ['cod_provincia' if nombre == 'cpro' else nombre for nombre in nombres]
diccionario.columns = nombres

diccionario['codigo_ine'] = diccionario['codigo_ine'].astype(str).str.zfill(5)

diccionario['codigo_ine'] = diccionario['codigo_ine'].astype(np.int64)
diccionario.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8132 entries, 0 to 8131
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   cod_ccaa       8132 non-null   int64 
 1   cod_provincia  8132 non-null   int64 
 2   cmun           8132 non-null   int64 
 3   dc             8132 non-null   int64 
 4   municipio      8132 non-null   object
 5   codigo_ine     8132 non-null   int64 
dtypes: int64(5), object(1)
memory usage: 381.3+ KB


In [ ]:
sociodem = pd.read_csv('/content/drive/MyDrive/Máster/TFM/columnas_sociodem.csv')
sociodem = sociodem.drop(index=11)
sociodem  =sociodem.reset_index(drop=True)
sociodem

,archivo,variables
0,V01_area_viaria.csv,"""pct_area_viaria"""
1,V02_pnc_subsidio_basico.csv,"""pnc_jubilacion_per_1000"", ""pnc_invalidez_per_..."
2,V03_densidad_poblacion.csv,"""densidad_poblacion_hab_km2"""
3,V04_tasa_crecimiento_poblacion.csv,"""tasa_crecimiento_pct"""
4,V05_clasificacion_urbano_rural.csv,"""clasificacion"""
5,V06_ratio_poblacion_area.csv,"""ratio_poblacion_area"""
6,V07_autonomia_financiera.csv,"""autonomia_financiera_pct"""
7,V08_residuos_domesticos_percapita.csv,"""residuos_kg_habitante"""
8,V09_generacion_aguas_residuales.csv,"""agua_suministrada_per_1000hab"""
9,V10_descarga_aguas_residuales.csv,"""agua_residual_per_1000hab"""


In [ ]:
ruta_general = '/content/drive/MyDrive/Máster/TFM/Irene Iribarren TFM (shared)/02_data/data/sociodemográficas/'

var_sociodemograficas = pd.DataFrame(data=None, columns=['cod_local', 'periodo', 'codigo_ine'])
anios = np.arange(2016, 2024, 1, dtype=int)

ids_usados = tabla['id']
localizaciones_usadas = maestro_geo[maestro_geo['cod_local'].isin(ids_usados)][['cod_local', 'cmun', 'cod_provincia', 'codigo_ine']]
localizaciones_usadas = pd.merge(localizaciones_usadas, diccionario, on=['cmun', 'cod_provincia', 'codigo_ine'], how='left').drop(columns='dc')
# print(localizaciones_usadas)
# print(localizaciones_usadas.info())
for n in range(len(sociodem)):

  #abrir el archivo
  archivo = pd.read_csv(f'{ruta_general}{sociodem['archivo'][n]}', dtype='object')

  #convertir la fecha a solo el año, sin las letras que tienen algunos datos
  if archivo['periodo'].dtype == 'O':
    archivo['periodo'] = archivo['periodo'].str.extract(r"(\d{4})")[0].astype(int)


  #separar las variables que hay anotadas en la tabla sociodem
  variables = sociodem['variables'][n].replace('"','').replace(",","").split()
  columnas_m2 = tabla.columns[tabla.columns.str.contains("_m2_", regex=False)].tolist()
  archivo[columnas_m2] = archivo[columnas_m2]/1000000
  nuevos_nombres = {
    columna: columna.replace(
        "_m2_",
        "_km2_"
    )
    for columna in columnas_m2}
  archivo = archivo.rename(columns=nuevos_nombres)

  #seleccionar las columnas que interesan, más el año y el codigo correspondiente
  if "codigo_ine" in archivo.columns:
    variables = ['codigo_ine', 'periodo'] + variables
    archivo_sel = archivo[variables]
    dicc = localizaciones_usadas[['codigo_ine', 'cod_local']]
    temp = pd.merge(dicc, archivo_sel, on='codigo_ine')
  elif "cod_provincia" in archivo.columns:
    variables = ['cod_provincia', 'periodo'] + variables
    archivo_sel = archivo[variables]
    dicc = localizaciones_usadas[['codigo_ine', 'cod_provincia', 'cod_local']]
    temp = pd.merge(dicc, archivo_sel, on='cod_provincia')
    temp = temp.drop(columns='cod_provincia')
  elif "cod_ccaa" in archivo.columns:
    variables = ['cod_ccaa', 'periodo'] + variables
    archivo_sel = archivo[variables]
    dicc = localizaciones_usadas[['codigo_ine', 'cod_ccaa', 'cod_local']]
    temp = pd.merge(dicc, archivo_sel, on='cod_ccaa')
    temp = temp.drop(columns='cod_ccaa')

  var_sociodemograficas = pd.merge(var_sociodemograficas, temp, on=['cod_local', 'periodo', 'codigo_ine'], how='outer')
var_sociodemograficas = var_sociodemograficas.drop_duplicates()
print(var_sociodemograficas.head())
#   print(temp.head(2))
#   print(temp.shape)
# print(archivo_sel.shape)

In [ ]:
nombre = 'sociodem_madrid.csv'
ruta = '/content/drive/MyDrive/Máster/TFM/Resultados/'
var_sociodemograficas.to_csv(f"{ruta}{nombre}", index=False, decimal = ',')
var_sociodemograficas

,cod_local,periodo,codigo_ine,pct_area_viaria,pnc_jubilacion_per_1000,pnc_invalidez_per_1000,pnc_total_per_1000,densidad_poblacion_hab_km2,tasa_crecimiento_pct,clasificacion,...,pct_forestal_osm,pct_parque,area_parque_m2_per_hab,area_verde_m2_per_hab,area_forestal_m2_per_hab,parques_per_1000hab,area_parque_m2_per_1000hab,pct_agua,pct_humedal,pct_ribera
